# Factor Investing Model — Interactive Walkthrough

This notebook walks through the full pipeline step by step:
1. Fetch data
2. Compute Value / Momentum / Size factors
3. Build a composite score and select a portfolio
4. Run a walk-forward backtest
5. Visualise performance

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from factor_investing.data import fetch_price_data, fetch_fundamental_data, get_sp500_tickers
from factor_investing.factors import ValueFactor, MomentumFactor, SizeFactor, FactorScorer
from factor_investing.portfolio import Backtester, compute_metrics
from factor_investing.visualization import plot_performance_dashboard, plot_factor_scores, plot_annual_returns

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Fetch Data

In [ ]:
START = '2018-01-01'
END   = '2024-12-31'

tickers = get_sp500_tickers()[:50]   # use 50 stocks for a quick demo
print(f'Universe: {len(tickers)} tickers')

prices = fetch_price_data(tickers, start=START, end=END)
print(f'Price panel: {prices.shape}')

In [ ]:
# Fundamentals (takes ~1 min for 50 tickers)
fundamentals = fetch_fundamental_data(list(prices.columns))
fundamentals[['market_cap','price_to_book','trailing_pe']].head(10)

## 2. Compute Individual Factor Scores

In [ ]:
value_factor    = ValueFactor(pb_weight=0.5, pe_weight=0.5)
momentum_factor = MomentumFactor(lookback_months=12, skip_months=1)
size_factor     = SizeFactor(use_log=True)

v_scores = value_factor.compute(fundamentals=fundamentals)
m_scores = momentum_factor.compute(prices=prices)
s_scores = size_factor.compute(fundamentals=fundamentals)

print('Value top 5:   ', v_scores.nlargest(5).index.tolist())
print('Momentum top 5:', m_scores.nlargest(5).index.tolist())
print('Size top 5:    ', s_scores.nlargest(5).index.tolist())

## 3. Composite Score & Portfolio Selection

In [ ]:
scorer = FactorScorer([
    (value_factor,    0.4),
    (momentum_factor, 0.4),
    (size_factor,     0.2),
])

scores = scorer.score(prices=prices, fundamentals=fundamentals)
scores.head(10)

In [ ]:
fig = plot_factor_scores(scores, top_n=15)
plt.show()

## 4. Walk-Forward Backtest

In [ ]:
spy = yf.download('SPY', start=START, end=END, auto_adjust=True, progress=False)['Close'].squeeze()

bt = Backtester(
    scorer=scorer,
    prices=prices,
    fundamentals=fundamentals,
    rebalance_freq='QS',     # quarterly rebalancing
    n_stocks=15,
    transaction_cost=0.001,
    benchmark_prices=spy,
)

results = bt.run()
results.head()

## 5. Performance Metrics

In [ ]:
port_metrics  = compute_metrics(results['portfolio'].dropna())
bench_metrics = compute_metrics(results['benchmark'].dropna())

comparison = pd.DataFrame({'Portfolio': port_metrics, 'SPY': bench_metrics})
comparison

## 6. Visualise

In [ ]:
fig = plot_performance_dashboard(
    portfolio_returns=results['portfolio'].dropna(),
    benchmark_returns=results['benchmark'].dropna(),
    metrics=port_metrics,
)
plt.show()

In [ ]:
fig = plot_annual_returns(
    portfolio_returns=results['portfolio'].dropna(),
    benchmark_returns=results['benchmark'].dropna(),
)
plt.show()

## 7. Rebalance Log
See which stocks were held at each rebalancing.

In [ ]:
bt.rebalance_log[['date','n_holdings','turnover']].tail(8)